# IGNIS Earth · Notebook 1: data → `celldays.parquet`

**What this notebook does.** It turns two free public datasets into one table that the model notebook (`02_model.ipynb`) can learn from:

* **NASA FIRMS VIIRS S-NPP** satellite fire detections for Greece, summers 2014 to 2025 (where fires burned).
* **Open-Meteo historical weather** (ERA5 reanalysis) and **Open-Meteo elevation** (Copernicus DEM).

**The output table** has one row per *candidate cell* per *fire-day*:

* Greece is cut into a grid of **0.05° cells (about 5 km)**, 110 rows × 135 columns.
* A **burning cell** is a cell with at least one cleaned VIIRS detection today.
* A **candidate cell** is any land cell within **2 rings (about 10 km)** of a burning cell, including the burning cells themselves. These are the only rows the model ever sees.
* The **target** `fire_tomorrow` is 1 if the candidate cell has a detection on the next fire-day.
* Headline task (**spread**): rows with `fire_today == 0`. Did the fire reach this cell by tomorrow? Rows with `fire_today == 1` are *continuing* fire and are reported separately.

**The fire-day.** The duty officer decides at 6 PM Athens time, so fire-day *d* runs from **18:00 on day d−1 to 18:00 on day d (Athens time)**. Fire detections and hourly weather are both cut on this window, so no feature uses anything the officer could not know at 6 PM.

**How to run.** `Runtime → Run all`. Every download is cached (to Google Drive if you allow it), so if Colab crashes or the weather rate limit stops you, just run all again and it continues where it stopped.

**Credits.** Code drafted with help from Claude (Anthropic) and reviewed and run by the team; list this in the write-up's Sources (Veritas rule 4).

## 0. Setup

In [ ]:
# global-land-mask tells us whether a point is on land or sea (no download needed).
!pip install -q global-land-mask

In [ ]:
import io, os, json, math, time, zipfile
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests
from scipy import ndimage
from global_land_mask import globe
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

### Settings
These are the only numbers you should need to change. Everything below reads them.

In [ ]:
# ---- Region: mainland Greece + Evia + Peloponnese (Crete and Aegean islands left out) ----
LAT_MIN, LAT_MAX = 36.25, 41.75
LON_MIN, LON_MAX = 20.00, 26.75
CELL = 0.05                       # grid cell size in degrees (about 5 km)

# ---- Years and the time split (never shuffle days) ----
YEARS       = list(range(2014, 2026))   # 2014 ... 2025
TRAIN_YEARS = list(range(2014, 2024))   # 2014 ... 2023
VAL_YEARS   = [2024]
TEST_YEARS  = [2025]                    # the model notebook touches this ONCE, after the freeze

# ---- Season ----
SEASON_START, SEASON_END = "06-01", "09-30"  # rows we keep (target for Sep 30 uses Oct 1)
FETCH_START,  FETCH_END  = "05-15", "10-01"  # data we download: 2 extra weeks before so
                                             # "fire in the last 7 days" and rain history exist on Jun 1

# ---- Fire-day window ----
TIMEZONE = "Europe/Athens"
DAY_CUTOFF_HOUR = 18              # fire-day d = 18:00 on d-1 to 18:00 on d, Athens time

# ---- Weather ----
WEATHER_STEP = 0.5                # weather point spacing in degrees. 0.5 = ~1-1.5 h download (paced
                                  # under the free limit). Set 1.0 if short on time (~20-30 min).
WEATHER_BATCH = 10                # points per Open-Meteo request
WEATHER_HOURLY_BUDGET = 4000      # Open-Meteo free limit is 5000 "calls" per hour; stay under it

# ---- Where files go ----
USE_DRIVE = True                  # save to Google Drive so nothing is lost if Colab resets

In [ ]:
# Pick the data folder. On Colab with USE_DRIVE it lives in your Drive: MyDrive/ignis_earth_data
DATA_DIR = Path("ignis_earth_data")
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path("/content/drive/MyDrive/ignis_earth_data")
    except ImportError:
        print("Not on Colab: saving to the local folder", DATA_DIR.resolve())
for sub in ["firms_raw", "weather_raw"]:
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Data folder:", DATA_DIR)

### The grid
Think of Greece as a picture of 110 × 135 pixels, where each pixel is one 5 km cell. Row 0 is the south edge, column 0 the west edge. We keep one such picture per day for "is it burning?", which lets us use simple image tools (blur, distance, connected blobs) to build the spread features.

In [ ]:
N_ROWS = round((LAT_MAX - LAT_MIN) / CELL)   # 110
N_COLS = round((LON_MAX - LON_MIN) / CELL)   # 135

KM_PER_DEG = 111.32
MID_LAT = (LAT_MIN + LAT_MAX) / 2
DY_KM = CELL * KM_PER_DEG                                  # height of a cell, ~5.57 km
DX_KM = CELL * KM_PER_DEG * math.cos(math.radians(MID_LAT))  # width of a cell, ~4.3 km

def lat_to_row(lat): return np.floor((np.asarray(lat) - LAT_MIN) / CELL + 1e-9).astype(int)
def lon_to_col(lon): return np.floor((np.asarray(lon) - LON_MIN) / CELL + 1e-9).astype(int)
def row_to_lat(row): return LAT_MIN + (np.asarray(row) + 0.5) * CELL   # cell centre
def col_to_lon(col): return LON_MIN + (np.asarray(col) + 0.5) * CELL

# Land mask: a cell is land if its centre is on land. We do NOT pick cells by where fires happened,
# because that would hide areas that burn for the first time.
_lat_grid, _lon_grid = np.meshgrid(row_to_lat(np.arange(N_ROWS)), col_to_lon(np.arange(N_COLS)), indexing="ij")
LAND = globe.is_land(_lat_grid, _lon_grid)

print(f"Grid: {N_ROWS} rows x {N_COLS} cols, cell = {DY_KM:.2f} km tall x {DX_KM:.2f} km wide")
print(f"Land cells: {LAND.sum():,} of {LAND.size:,}")
plt.figure(figsize=(5, 4)); plt.imshow(LAND, origin="lower", cmap="Greens",
           extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]); plt.title("Land mask"); plt.show()

### Wind alignment (and its unit test)
`wind_alignment` says whether a cell is **downwind** of a fire: the cosine of the angle between *where the wind blows toward* and *the direction from the fire to the cell*. +1 = directly downwind, 0 = side wind, −1 = directly upwind.

Open-Meteo reports wind direction as where the wind **comes from** (meteorology convention), so we add 180°.

This is the Earth version of the IGNIS idea of *concurrent vs opposed flow*: flame spreads faster with the flow than against it.

In [ ]:
def wind_alignment(fire_lat, fire_lon, cell_lat, cell_lon, wind_from_deg):
    # Works on single numbers or whole columns at once.
    dy = (np.asarray(cell_lat) - fire_lat) * KM_PER_DEG                                  # km north
    dx = (np.asarray(cell_lon) - fire_lon) * KM_PER_DEG * np.cos(np.radians((np.asarray(cell_lat) + fire_lat) / 2))  # km east
    dist = np.hypot(dx, dy)
    toward = np.radians((np.asarray(wind_from_deg) + 180.0) % 360.0)   # direction the wind blows toward
    wx, wy = np.sin(toward), np.cos(toward)                            # compass: 0 = north, 90 = east
    with np.errstate(invalid="ignore", divide="ignore"):
        cos_angle = (dx * wx + dy * wy) / dist
    return np.where(dist > 0, cos_angle, 0.0)   # a burning cell vs itself has no direction -> 0

# Unit test (from the plan). Fire one cell NORTH of the candidate cell:
lat0, lon0 = 38.0, 23.0
fire_lat = lat0 + CELL
assert np.isclose(wind_alignment(fire_lat, lon0, lat0, lon0, 0.0), 1.0)     # wind FROM north -> cell is downwind
assert np.isclose(wind_alignment(fire_lat, lon0, lat0, lon0, 180.0), -1.0)  # wind FROM south -> cell is upwind
assert abs(wind_alignment(fire_lat, lon0, lat0, lon0, 90.0)) < 1e-9         # wind from east -> side wind
# Fire one cell WEST, wind from the west (270) blows east -> downwind
assert np.isclose(wind_alignment(lat0, lon0 - CELL, lat0, lon0, 270.0), 1.0)
# Ring offsets: 2 rings of 0.05 deg = a 5x5 block, 0.25 deg wide
assert np.ones((5, 5)).shape[0] * CELL == 0.25
print("wind_alignment unit tests passed")

## 1. Download NASA FIRMS fire detections

**Route A (no key):** FIRMS publishes one ready-made CSV per country per year ("Yearly summary by country"). We try that first.

**Route B (needs a free MAP_KEY):** if a year's file is missing, we ask the FIRMS area API for our bounding box, 5 days per request. Get a key in one minute at <https://firms.modaps.eosdis.nasa.gov/api/map_key/> (enter your email; the key arrives by email). In Colab, store it under the 🔑 *Secrets* tab as `FIRMS_MAP_KEY`, or paste it when asked.

**Route C (manual):** if you downloaded the CSVs yourself, put them in `ignis_earth_data/firms_raw/` named `firms_2014.csv` ... `firms_2025.csv` and they are used as-is.

We use **VIIRS S-NPP only** (375 m pixels, day and night passes), so every season comes from the same sensor.

In [ ]:
COUNTRY_URL = "https://firms.modaps.eosdis.nasa.gov/data/country/viirs-snpp/{year}/viirs-snpp_{year}_Greece.csv"
API_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{source}/{area}/{days}/{date}"
_MAP_KEY = None

def get_map_key():
    global _MAP_KEY
    if _MAP_KEY:
        return _MAP_KEY
    try:
        from google.colab import userdata
        _MAP_KEY = userdata.get("FIRMS_MAP_KEY")
    except Exception:
        _MAP_KEY = os.environ.get("FIRMS_MAP_KEY") or getpass("Paste your FIRMS MAP_KEY: ").strip()
    return _MAP_KEY

def download_country_file(year, path):
    # Route A: the yearly Greece CSV. Returns True if it worked.
    try:
        r = requests.get(COUNTRY_URL.format(year=year), timeout=180)
    except requests.RequestException as e:
        print(f"  {year}: country file failed ({e})")
        return False
    if r.status_code != 200 or not r.text.startswith("latitude"):
        print(f"  {year}: no country file (HTTP {r.status_code})")
        return False
    path.write_text(r.text)
    return True

def download_with_api(year, path):
    # Route B: area API, 5 days at a time. SP = standard (archive) data; NRT = near-real-time,
    # only used if the archive has nothing for that window yet.
    key = get_map_key()
    area = f"{LON_MIN},{LAT_MIN},{LON_MAX},{LAT_MAX}"         # west,south,east,north
    day = pd.Timestamp(f"{year}-{FETCH_START}") - pd.Timedelta(days=1)  # 1 extra day for the UTC->Athens shift
    end = pd.Timestamp(f"{year}-{FETCH_END}")
    parts = []
    while day <= end:
        for source in ["VIIRS_SNPP_SP", "VIIRS_SNPP_NRT"]:
            url = API_URL.format(key=key, source=source, area=area, days=5, date=day.strftime("%Y-%m-%d"))
            text = requests.get(url, timeout=180).text
            if not text.startswith("latitude"):
                raise RuntimeError(f"FIRMS API said: {text[:200]!r} (check your MAP_KEY)")
            chunk = pd.read_csv(io.StringIO(text), dtype={"acq_time": str})
            if len(chunk) > 0:
                parts.append(chunk)
                break
        day += pd.Timedelta(days=5)
        time.sleep(0.5)  # be polite
    df = pd.concat(parts, ignore_index=True).drop_duplicates() if parts else pd.DataFrame()
    df.to_csv(path, index=False)
    return len(df) > 0

for year in YEARS:
    path = DATA_DIR / "firms_raw" / f"firms_{year}.csv"
    if path.exists() and path.stat().st_size > 100:
        print(f"{year}: cached")
        continue
    print(f"{year}: downloading ...")
    if not download_country_file(year, path):
        print(f"  {year}: trying the area API (needs MAP_KEY)")
        download_with_api(year, path)
    print(f"  {year}: saved {path.stat().st_size/1e6:.1f} MB")

## 2. Clean the detections and give each one a fire-day

Cleaning steps (from the plan):
1. Keep points inside the bounding box.
2. Drop **low confidence** detections (`confidence == "l"`).
3. Keep **`type == 0`** (presumed vegetation fire). This drops volcanoes, gas flares and industrial heat sources that FIRMS already flags. (The yearly country files have `type`; the API files do not, so then this step is skipped and we say so.)
4. Convert `acq_date` + `acq_time` from **UTC to Athens time**, then assign the fire-day: a detection at 01:30 AM Sep 1 belongs to fire-day Sep 1; one at 19:00 Sep 1 belongs to fire-day Sep 2.
5. Snap each point to its 5 km cell (row, column).

In [ ]:
def clean_year(year):
    df = pd.read_csv(DATA_DIR / "firms_raw" / f"firms_{year}.csv", dtype={"acq_time": str})
    n_raw = len(df)
    # 1. bounding box
    df = df[df.latitude.between(LAT_MIN, LAT_MAX, inclusive="left") &
            df.longitude.between(LON_MIN, LON_MAX, inclusive="left")]
    # 2. confidence: VIIRS uses l / n / h (some files spell it out: low / nominal / high)
    df = df[df.confidence.astype(str).str.lower().str[0] != "l"]
    # 3. vegetation fires only
    has_type = "type" in df.columns
    if has_type:
        df = df[df["type"] == 0]
    # 4. UTC -> Athens time -> fire-day
    hhmm = df.acq_time.astype(str).str.replace(":", "").str.zfill(4)
    utc = pd.to_datetime(df.acq_date.astype(str) + " " + hhmm, format="%Y-%m-%d %H%M", utc=True)
    local = utc.dt.tz_convert(TIMEZONE).dt.tz_localize(None)     # Athens wall-clock time
    fire_day = (local + pd.Timedelta(hours=24 - DAY_CUTOFF_HOUR)).dt.normalize()
    out = pd.DataFrame({
        "year": year,
        "fire_day": fire_day.values,
        "local_time": local.values,
        "row": lat_to_row(df.latitude.values),
        "col": lon_to_col(df.longitude.values),
        "frp": df.frp.values,
        "daynight": df.daynight.values,
    })
    # keep only our download window (May 15 to Oct 1 of that year)
    out = out[out.fire_day.between(pd.Timestamp(f"{year}-{FETCH_START}"), pd.Timestamp(f"{year}-{FETCH_END}"))]
    print(f"{year}: {n_raw:>7,} raw -> {len(out):>6,} clean in window" + ("" if has_type else "  (no 'type' column: static-source filter skipped)"))
    return out

fires = pd.concat([clean_year(y) for y in YEARS], ignore_index=True)
fires.to_parquet(DATA_DIR / "fires_clean.parquet", index=False)
print(f"\nTotal clean detections: {len(fires):,}")
print("Detections by local hour (should cluster near ~1-3 AM and ~12-2 PM, the S-NPP passes):")
print(fires.local_time.dt.hour.value_counts().sort_index().to_dict())

## 3. One row per burning cell per day

Group the detections by (fire-day, cell): how many detections and the total fire radiative power (FRP, megawatts), our measure of fire intensity.

In [ ]:
fire_cells = (fires.groupby(["fire_day", "row", "col"])
                   .agg(det_count=("frp", "size"), frp_sum=("frp", "sum"))
                   .reset_index())
fire_cells["year"] = fire_cells.fire_day.dt.year
fire_cells.to_parquet(DATA_DIR / "fire_cells.parquet", index=False)   # the demo uses this for maps and the "reveal"

in_season = fire_cells[fire_cells.fire_day.dt.strftime("%m-%d").between(SEASON_START, SEASON_END)]
print("Burning cell-days per season (Jun-Sep):")
print(in_season.groupby("year").size().to_string())

## 4. Candidate cells, target, and the fire-based spread features

For every fire-day in the season we take that day's "burning" picture and compute, for each candidate cell:

| Feature | Meaning |
|---|---|
| `fire_today`, `det_count_today`, `frp_today` | is the cell burning, log(1 + detections), log(1 + FRP) |
| `fire_last_3d`, `fire_last_7d` | days with fire in this cell in the 3 / 7 days *before* today |
| `n_fire_1ring`, `n_fire_2ring` | burning cells among the 8 neighbours / the 24 cells within 2 rings (cell itself not counted) |
| `frp_nearby` | log(1 + FRP of all burning cells within 2 rings, cell included) |
| `dist_to_fire` | km to the nearest burning cell (0 if burning) |
| `fire_age` | days in a row that the nearest burning cell (or a neighbour of it) has had fire |
| `cluster_size` | number of connected burning cells in the nearest fire |
| `fire_id` | an id for the nearest fire that day (used by the demo's fire dropdown and the per-fire direction metric) |
| `fire_tomorrow` | **the target**: does this cell have a detection on the next fire-day? |

We also keep `near_row`, `near_col` (the nearest burning cell) and `nbr_mask` (which of the 8 neighbours burned), so that once the weather is in we can compute downwind and uphill.

In [ ]:
K3 = np.ones((3, 3), dtype=int)                 # 1 ring
K5 = np.ones((5, 5), dtype=int)                 # 2 rings
OFFSETS8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]   # (d_row, d_col)

def shift(grid, dr, dc):
    # value of the neighbour at (row+dr, col+dc) for every cell; outside the map = 0
    p = np.pad(grid, 1)
    return p[1 + dr: 1 + dr + N_ROWS, 1 + dc: 1 + dc + N_COLS]

def build_rows_for_year(year):
    days = pd.date_range(f"{year}-{FETCH_START}", f"{year}-{FETCH_END}")
    nd = len(days)
    FIRE = np.zeros((nd, N_ROWS, N_COLS), dtype=bool)
    FRP = np.zeros((nd, N_ROWS, N_COLS))
    DET = np.zeros((nd, N_ROWS, N_COLS))
    fc = fire_cells[fire_cells.year == year]
    t_idx = (fc.fire_day - days[0]).dt.days.values
    FIRE[t_idx, fc.row.values, fc.col.values] = True
    FRP[t_idx, fc.row.values, fc.col.values] = fc.frp_sum.values
    DET[t_idx, fc.row.values, fc.col.values] = fc.det_count.values

    season = (days.strftime("%m-%d") >= SEASON_START) & (days.strftime("%m-%d") <= SEASON_END)
    streak = np.zeros((N_ROWS, N_COLS), dtype=int)
    out = []
    for t, day in enumerate(days):
        fire = FIRE[t]
        # fire_age bookkeeping runs every day (also before the season starts)
        active = ndimage.binary_dilation(fire, structure=K3)
        streak = np.where(active, streak + 1, 0)
        if not season[t] or t + 1 >= nd or not fire.any():
            continue

        cand = ndimage.binary_dilation(fire, structure=K5) & LAND    # land cells within 2 rings
        f_int = fire.astype(int)
        n1 = ndimage.convolve(f_int, K3, mode="constant") - f_int
        n2 = ndimage.convolve(f_int, K5, mode="constant") - f_int
        frp_near = ndimage.convolve(FRP[t], K5.astype(float), mode="constant")
        last3 = FIRE[max(0, t - 3):t].sum(axis=0)
        last7 = FIRE[max(0, t - 7):t].sum(axis=0)
        # distance (km) to the nearest burning cell, and which cell that is
        dist, (near_r, near_c) = ndimage.distance_transform_edt(~fire, sampling=(DY_KM, DX_KM), return_indices=True)
        # connected fires (8-connectivity): each blob of touching burning cells is one fire
        labels, _ = ndimage.label(fire, structure=K3)
        sizes = np.bincount(labels.ravel())
        # bitmask of which neighbours are burning (bit k = OFFSETS8[k])
        nbr_mask = np.zeros((N_ROWS, N_COLS), dtype=int)
        for k, (dr, dc) in enumerate(OFFSETS8):
            nbr_mask |= shift(f_int, dr, dc) << k

        rr, cc = np.nonzero(cand)
        nr, nc = near_r[rr, cc], near_c[rr, cc]
        lab = labels[nr, nc]
        out.append(pd.DataFrame({
            "day": day, "row": rr, "col": cc,
            "fire_today": fire[rr, cc].astype(int),
            "det_count_today": np.log1p(DET[t][rr, cc]),
            "frp_today": np.log1p(FRP[t][rr, cc]),
            "fire_last_3d": last3[rr, cc], "fire_last_7d": last7[rr, cc],
            "n_fire_1ring": n1[rr, cc], "n_fire_2ring": n2[rr, cc],
            "frp_nearby": np.log1p(frp_near[rr, cc]),
            "dist_to_fire": dist[rr, cc],
            "near_row": nr, "near_col": nc,
            "fire_age": streak[nr, nc],
            "cluster_size": sizes[lab],
            "fire_id": [f"{day:%Y%m%d}_{l}" for l in lab],
            "nbr_mask": nbr_mask[rr, cc],
            "fire_tomorrow": FIRE[t + 1][rr, cc].astype(int),   # the target
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

celldays = pd.concat([build_rows_for_year(y) for y in YEARS], ignore_index=True)
celldays["year"] = celldays.day.dt.year
celldays["lat"] = row_to_lat(celldays.row)
celldays["lon"] = col_to_lon(celldays.col)
print(f"Candidate cell-days: {len(celldays):,}")
celldays.head()

### ✅ The 8 PM gate: are there enough spread events?
The plan's rule: if the **2024 validation season has fewer than about 100 newly reached cells**, switch to the California fallback (same pipeline, new box and season). Check the 2025 test season too.

In [ ]:
celldays["newly_reached"] = ((celldays.fire_today == 0) & (celldays.fire_tomorrow == 1)).astype(int)
celldays["continuing"] = ((celldays.fire_today == 1) & (celldays.fire_tomorrow == 1)).astype(int)
gate = celldays.groupby("year").agg(
    candidate_rows=("row", "size"),
    burning_today=("fire_today", "sum"),
    newly_reached=("newly_reached", "sum"),
    continuing=("continuing", "sum"),
    fires=("fire_id", "nunique"),
)
gate["spread_rate_%"] = (100 * gate.newly_reached / (gate.candidate_rows - gate.burning_today)).round(2)
print(gate.to_string())
for y in VAL_YEARS + TEST_YEARS:
    n = int(gate.newly_reached.get(y, 0))
    print(f"{y}: {n} newly reached cells -> " + ("OK" if n >= 100 else "BELOW 100: talk to the team about the fallback"))

In [ ]:
# Sanity picture: the biggest fire-day of 2024. Red = burning today, grey = candidates,
# orange = candidates that burned tomorrow (newly reached = what we want to predict).
y = VAL_YEARS[0] if (celldays.year == VAL_YEARS[0]).any() else celldays.year.max()
d = celldays[celldays.year == y].groupby("day").fire_today.sum().idxmax()
one = celldays[celldays.day == d]
plt.figure(figsize=(7, 6))
plt.imshow(LAND, origin="lower", cmap="Greys", alpha=0.25, extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
plt.scatter(one.lon, one.lat, s=6, c="lightgrey", label="candidate cells")
plt.scatter(one[one.newly_reached == 1].lon, one[one.newly_reached == 1].lat, s=10, c="orange", label="newly reached tomorrow")
plt.scatter(one[one.fire_today == 1].lon, one[one.fire_today == 1].lat, s=10, c="red", label="burning today")
plt.title(f"Fire-day {d:%Y-%m-%d}"); plt.legend(loc="lower left"); plt.show()

## 5. Weather from Open-Meteo (hourly, cut on the 6 PM window)

* We only download weather points that are actually needed: the nearest `WEATHER_STEP` grid point of every candidate cell.
* Open-Meteo's **daily** values run midnight to midnight, which would leak the evening's wind and rain into "today". So we download **hourly** values in Athens time and aggregate them ourselves over 18:00 → 18:00.
* **Free-tier limits:** each request "costs" `locations × days/14 × variables/10`, against 5,000 per hour and 10,000 per day. The loop below keeps a running total and pauses when it gets close, so it may sit and wait. That is normal. Every point-year is saved as soon as it arrives, so re-running continues where it stopped.

In [ ]:
HOURLY_VARS = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_gusts_10m",
               "wind_direction_10m", "precipitation", "vapour_pressure_deficit", "et0_fao_evapotranspiration"]
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

# Which weather point does each candidate cell use? (nearest point on a WEATHER_STEP grid)
celldays["w_lat"] = (np.round(celldays.lat / WEATHER_STEP) * WEATHER_STEP).round(3)
celldays["w_lon"] = (np.round(celldays.lon / WEATHER_STEP) * WEATHER_STEP).round(3)
points = celldays[["w_lat", "w_lon"]].drop_duplicates().sort_values(["w_lat", "w_lon"]).reset_index(drop=True)
n_days = (pd.Timestamp(f"2000-{FETCH_END}") - pd.Timestamp(f"2000-{FETCH_START}")).days + 1
total_cost = len(points) * len(YEARS) * (n_days / 14) * (len(HOURLY_VARS) / 10)
print(f"{len(points)} weather points x {len(YEARS)} years. Estimated cost {total_cost:,.0f} "
      f"(free: 5,000/hour, 10,000/day) -> about {total_cost / WEATHER_HOURLY_BUDGET:.1f} h at our pace")

In [ ]:
WEATHER_DIR = DATA_DIR / "weather_raw" / f"step{WEATHER_STEP}"
WEATHER_DIR.mkdir(parents=True, exist_ok=True)
def point_file(lat, lon, year): return WEATHER_DIR / f"{lat:.3f}_{lon:.3f}_{year}.parquet"

_spent = []   # (time, cost) of requests in the last hour, for pacing
def wait_for_budget(cost):
    while True:
        now = time.time()
        _spent[:] = [(t, c) for t, c in _spent if now - t < 3600]
        if sum(c for _, c in _spent) + cost <= WEATHER_HOURLY_BUDGET:
            _spent.append((now, cost)); return
        wait = 3600 - (now - _spent[0][0]) + 5
        print(f"   pausing {wait/60:.0f} min to stay under the hourly limit ...")
        time.sleep(min(wait, 600))

def fetch_batch(batch, year):
    start, end = f"{year}-{FETCH_START}", f"{year}-{FETCH_END}"
    params = {"latitude": ",".join(f"{v:.3f}" for v in batch.w_lat),
              "longitude": ",".join(f"{v:.3f}" for v in batch.w_lon),
              "start_date": start, "end_date": end,
              "hourly": ",".join(HOURLY_VARS), "timezone": TIMEZONE}
    cost = len(batch) * (n_days / 14) * (len(HOURLY_VARS) / 10)
    for attempt in range(8):
        wait_for_budget(cost)
        try:
            r = requests.get(ARCHIVE_URL, params=params, timeout=180)
        except requests.RequestException as e:
            print("   network error, retrying:", e); time.sleep(20 * (attempt + 1)); continue
        if r.status_code == 200:
            data = r.json()
            return data if isinstance(data, list) else [data]   # one location -> a dict, several -> a list
        reason = r.text[:200]
        if r.status_code == 429 and "daily" in reason.lower():
            raise RuntimeError("Open-Meteo daily limit reached. Re-run this cell tomorrow (cached data is kept) "
                               "or set WEATHER_STEP = 1.0 and re-run.")
        print(f"   HTTP {r.status_code}: {reason} -> waiting and retrying")
        time.sleep(60 * (attempt + 1))
    raise RuntimeError("Open-Meteo kept failing; re-run this cell later (cached data is kept).")

for year in YEARS:
    todo = points[[not point_file(a, b, year).exists() for a, b in zip(points.w_lat, points.w_lon)]]
    print(f"{year}: {len(points) - len(todo)} cached, {len(todo)} to download")
    for i in range(0, len(todo), WEATHER_BATCH):
        batch = todo.iloc[i:i + WEATHER_BATCH]
        for (lat, lon), loc in zip(zip(batch.w_lat, batch.w_lon), fetch_batch(batch, year)):
            h = pd.DataFrame(loc["hourly"])
            h["time"] = pd.to_datetime(h["time"])
            h["w_lat"], h["w_lon"] = lat, lon
            h.to_parquet(point_file(lat, lon, year), index=False)
print("Weather download complete")

### Aggregate hourly weather to fire-days
For each weather point and fire-day (18:00 → 18:00): max temperature, min humidity, max wind and gusts, the **dominant wind direction** (the average of hourly directions weighted by wind speed, done with sine and cosine so 350° and 10° average to 0°, not 180°), rain sum, evapotranspiration sum and max vapour pressure deficit. Then the history features: 7-day rain, 7-day evapotranspiration and days since the last day with at least 1 mm of rain.

In [ ]:
hourly = pd.concat([pd.read_parquet(point_file(a, b, y)) for y in YEARS
                    for a, b in zip(points.w_lat, points.w_lon)], ignore_index=True)
hourly["day"] = (hourly.time + pd.Timedelta(hours=24 - DAY_CUTOFF_HOUR)).dt.normalize()
rad = np.radians(hourly.wind_direction_10m)
hourly["wx"] = hourly.wind_speed_10m * np.sin(rad)
hourly["wy"] = hourly.wind_speed_10m * np.cos(rad)

wd = (hourly.groupby(["w_lat", "w_lon", "day"])
            .agg(tmax=("temperature_2m", "max"), rh_min=("relative_humidity_2m", "min"),
                 wind_max=("wind_speed_10m", "max"), gust_max=("wind_gusts_10m", "max"),
                 wx=("wx", "sum"), wy=("wy", "sum"),
                 rain_today=("precipitation", "sum"), et0_today=("et0_fao_evapotranspiration", "sum"),
                 vpd_max=("vapour_pressure_deficit", "max"), n_hours=("time", "size"))
            .reset_index())
wd["wind_dir"] = np.degrees(np.arctan2(wd.wx, wd.wy)) % 360      # dominant direction the wind comes FROM
wd = wd.drop(columns=["wx", "wy"])
wd["year"] = wd.day.dt.year

# history features, separately for each point and year
wd = wd.sort_values(["w_lat", "w_lon", "day"]).reset_index(drop=True)
g = wd.groupby(["w_lat", "w_lon", "year"])
wd["rain_7d"] = g.rain_today.transform(lambda s: s.rolling(7, min_periods=1).sum())
wd["et0_7d"] = g.et0_today.transform(lambda s: s.rolling(7, min_periods=1).sum())
rainy = (wd.rain_today >= 1.0).astype(int)
spell = rainy.groupby([wd.w_lat, wd.w_lon, wd.year]).cumsum()          # a new "spell" starts on each rainy day
wd["days_since_rain"] = wd.groupby([wd.w_lat, wd.w_lon, wd.year, spell]).cumcount()
# (before the first rain after May 15 this counts from May 15, so it is a lower bound; say so in the write-up)

print("Hours per fire-day (should be 24, except the first and DST days):", wd.n_hours.value_counts().head(4).to_dict())
wd.to_parquet(DATA_DIR / "weather_daily.parquet", index=False)
wd.describe().T[["mean", "min", "max"]].round(1)

## 6. Elevation for every cell we need (Open-Meteo Elevation API, Copernicus DEM ~90 m)
Needed for **upslope**: fires run faster uphill. We query each cell's centre once (100 cells per request) and cache the result.

In [ ]:
ELEV_URL = "https://api.open-meteo.com/v1/elevation"
elev_path = DATA_DIR / "cell_elevation.parquet"
need = pd.concat([celldays[["row", "col"]],
                  celldays[["near_row", "near_col"]].set_axis(["row", "col"], axis=1)]).drop_duplicates()
elev = (pd.read_parquet(elev_path) if elev_path.exists()
        else pd.DataFrame({"row": pd.Series(dtype=int), "col": pd.Series(dtype=int), "elev": pd.Series(dtype=float)}))
todo = need.merge(elev[["row", "col"]], how="left", indicator=True).query("_merge == 'left_only'")[["row", "col"]]
print(f"{len(need):,} cells need elevation, {len(todo):,} still to download")

new = []
for i in range(0, len(todo), 100):
    chunk = todo.iloc[i:i + 100]
    params = {"latitude": ",".join(f"{v:.3f}" for v in row_to_lat(chunk.row)),
              "longitude": ",".join(f"{v:.3f}" for v in col_to_lon(chunk.col))}
    for attempt in range(5):
        r = requests.get(ELEV_URL, params=params, timeout=60)
        if r.status_code == 200: break
        print("   HTTP", r.status_code, r.text[:120]); time.sleep(30 * (attempt + 1))
    r.raise_for_status()
    new.append(chunk.assign(elev=r.json()["elevation"]))
    if len(new) % 20 == 0:     # save progress every 2,000 cells
        elev = pd.concat([elev] + new, ignore_index=True); new = []; elev.to_parquet(elev_path, index=False)
    time.sleep(0.3)
elev = pd.concat([elev] + new, ignore_index=True)
elev[["row", "col"]] = elev[["row", "col"]].astype(int)
elev["elev"] = elev.elev.astype(float)
elev.to_parquet(elev_path, index=False)
print("Elevation range (m):", elev.elev.min(), "to", elev.elev.max())

## 7. Put it together: weather, downwind, uphill and Greece-specific features

| Feature | How it is computed | IGNIS link |
|---|---|---|
| `tmax`, `rh_min`, `vpd_max` | max temperature (°C), min humidity (%), max vapour pressure deficit (kPa), 18:00→18:00 | fuel heating and dryness |
| `wind_max`, `gust_max` | max wind and gusts (km/h) | forced flow velocity |
| `wind_dir_sin`, `wind_dir_cos` | dominant wind direction | flow direction |
| `rain_today`, `rain_7d`, `days_since_rain`, `et0_7d` | rain and drying history | fuel moisture |
| `wind_alignment` | +1 if the cell is directly downwind of the nearest burning cell | concurrent vs opposed flow |
| `upwind_nbr_fire` | 1 if a burning neighbour sits upwind of this cell (alignment > 0.5) | concurrent flow |
| `elev`, `upslope` | cell elevation (m); (cell elev − nearest fire elev) / distance, m per km | uphill spread |
| `meltemi` | wind from 340°–70° (N to NE) **and** gusts ≥ 50 km/h | Greece: Etesian winds |
| `gust_x_downwind` | `gust_max × wind_alignment` | strong gusts matter most downwind |
| `doy_sin`, `doy_cos` | day of year | season |
| `cell_clim` | the cell's fire-day rate in that month, **training years only** | local fuel and human activity |

Land-cover features (`frac_conifer`, `frac_maquis`, ...) and `rh_spotting` need CORINE and are Sunday add-ons.

In [ ]:
df = celldays.merge(wd.drop(columns=["year", "n_hours"]), on=["w_lat", "w_lon", "day"], how="left")
df = df.merge(elev, on=["row", "col"], how="left")
df = df.merge(elev.rename(columns={"row": "near_row", "col": "near_col", "elev": "near_elev"}),
              on=["near_row", "near_col"], how="left")

# Downwind: from the nearest burning cell to this cell
near_lat, near_lon = row_to_lat(df.near_row.values), col_to_lon(df.near_col.values)
df["wind_alignment"] = wind_alignment(near_lat, near_lon, df.lat.values, df.lon.values, df.wind_dir.values)

# Upwind neighbour on fire: check each of the 8 neighbours that is burning
upwind = np.zeros(len(df), dtype=bool)
for k, (dr, dc) in enumerate(OFFSETS8):
    burning = (df.nbr_mask.values >> k) & 1 == 1
    a = wind_alignment(df.lat.values + dr * CELL, df.lon.values + dc * CELL, df.lat.values, df.lon.values, df.wind_dir.values)
    upwind |= burning & (a > 0.5)
df["upwind_nbr_fire"] = upwind.astype(int)

# Uphill: metres of climb per km from the fire to this cell
df["upslope"] = np.where(df.dist_to_fire > 0, (df.elev - df.near_elev) / df.dist_to_fire.where(df.dist_to_fire > 0), 0.0)

# Direction and season as sine/cosine, so 359 and 1 degrees are close
df["wind_dir_sin"] = np.sin(np.radians(df.wind_dir)); df["wind_dir_cos"] = np.cos(np.radians(df.wind_dir))
doy = df.day.dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25); df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

# Greece-specific
df["meltemi"] = (((df.wind_dir >= 340) | (df.wind_dir <= 70)) & (df.gust_max >= 50)).astype(int)
df["gust_x_downwind"] = df.gust_max * df.wind_alignment

# Climatology from TRAINING YEARS ONLY (using 2024/2025 here would leak the answer)
train_fc = in_season[in_season.year.isin(TRAIN_YEARS)].copy()
train_fc["month"] = train_fc.fire_day.dt.month
season_days = pd.Series(pd.date_range(f"2001-{SEASON_START}", f"2001-{SEASON_END}").month).value_counts()
clim = train_fc.groupby(["row", "col", "month"]).size().rename("n").reset_index()
clim["cell_clim"] = clim.n / (clim.month.map(season_days) * len(TRAIN_YEARS))
df["month"] = df.day.dt.month
df = df.merge(clim[["row", "col", "month", "cell_clim"]], on=["row", "col", "month"], how="left")
df["cell_clim"] = df.cell_clim.fillna(0.0)

# Which split each row belongs to
df["split"] = np.select([df.year.isin(TRAIN_YEARS), df.year.isin(VAL_YEARS), df.year.isin(TEST_YEARS)],
                        ["train", "val", "test"], "other")
print(df.split.value_counts().to_string())

## 8. Checks, then save `celldays.parquet`

In [ ]:
FEATURES = ["fire_today", "det_count_today", "frp_today", "fire_last_3d", "fire_last_7d",
            "n_fire_1ring", "n_fire_2ring", "frp_nearby", "dist_to_fire", "fire_age", "cluster_size",
            "wind_alignment", "upwind_nbr_fire", "upslope", "elev",
            "tmax", "rh_min", "wind_max", "gust_max", "wind_dir_sin", "wind_dir_cos", "vpd_max",
            "rain_today", "rain_7d", "days_since_rain", "et0_7d", "meltemi", "gust_x_downwind",
            "doy_sin", "doy_cos", "cell_clim"]
ID_COLS = ["day", "year", "split", "row", "col", "lat", "lon", "fire_id", "near_row", "near_col", "wind_dir"]
TARGET = "fire_tomorrow"

# 1. Missing values (weather or elevation gaps show up here)
missing = df[FEATURES].isna().mean().mul(100).round(2)
print("Missing values (% of rows), only columns with any:"); print(missing[missing > 0].to_string() or "  none")

# 2. The target really is TOMORROW's fire: check 1,000 random rows against fire_cells
burn_set = set(zip(fire_cells.fire_day, fire_cells.row, fire_cells.col))
s = df.sample(min(1000, len(df)), random_state=0)
check = [((d + pd.Timedelta(days=1), r, c) in burn_set) == bool(t) for d, r, c, t in zip(s.day, s.row, s.col, s.fire_tomorrow)]
assert all(check), "target does not match next-day detections!"
# 3. fire_today matches today's detections
check = [((d, r, c) in burn_set) == bool(t) for d, r, c, t in zip(s.day, s.row, s.col, s.fire_today)]
assert all(check), "fire_today does not match today's detections!"
# 4. Every row is a land cell within 2 rings of a fire, inside the season
assert LAND[df.row, df.col].all()
assert (df.dist_to_fire <= 2.01 * math.hypot(DY_KM, DX_KM)).all()
assert df.day.dt.strftime("%m-%d").between(SEASON_START, SEASON_END).all()
print("Target, fire_today, land and ring checks passed")

print(f"\nRows: {len(df):,}   spread rows (fire_today=0): {(df.fire_today == 0).sum():,}   "
      f"newly reached: {df.newly_reached.sum():,}")

In [ ]:
out = df[ID_COLS + FEATURES + [TARGET, "newly_reached"]].copy()
out.to_parquet(DATA_DIR / "celldays.parquet", index=False)
print("Saved", DATA_DIR / "celldays.parquet", f"({len(out):,} rows, {len(out.columns)} columns)")
out.groupby("split")[["fire_today", TARGET, "newly_reached"]].mean().round(4)

In [ ]:
# Quick look for the team: does spread really go downwind? (It should, or something is wrong.)
spread_rows = out[out.fire_today == 0]
print("Chance a non-burning candidate cell burns tomorrow, by wind alignment:")
print(spread_rows.groupby(pd.cut(spread_rows.wind_alignment, [-1.01, -0.5, 0, 0.5, 1.01]), observed=True)[TARGET]
      .agg(["mean", "size"]).round(3).to_string())
print("\n... and by distance to the fire (km):")
print(spread_rows.groupby(pd.cut(spread_rows.dist_to_fire, [0, 5, 8, 12]), observed=True)[TARGET]
      .agg(["mean", "size"]).round(3).to_string())

## 9. (Optional, 15 min) Villages and towns for the demo's "villages at risk" list
GeoNames populated places (feature class `P`), free under CC BY. Each place is snapped to its 5 km cell.

In [ ]:
vill_path = DATA_DIR / "villages.parquet"
if not vill_path.exists():
    r = requests.get("https://download.geonames.org/export/dump/GR.zip", timeout=300)
    r.raise_for_status()
    cols = ["geonameid", "name", "asciiname", "alternatenames", "latitude", "longitude", "feature_class",
            "feature_code", "country_code", "cc2", "admin1", "admin2", "admin3", "admin4", "population",
            "elevation", "dem", "timezone", "modified"]
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        gn = pd.read_csv(z.open("GR.txt"), sep="\t", header=None, names=cols, low_memory=False, keep_default_na=False)
    gn = gn[(gn.feature_class == "P") & gn.latitude.between(LAT_MIN, LAT_MAX, inclusive="left")
            & gn.longitude.between(LON_MIN, LON_MAX, inclusive="left")]
    vill = gn[["geonameid", "name", "asciiname", "latitude", "longitude", "population"]].copy()
    vill["population"] = pd.to_numeric(vill.population, errors="coerce").fillna(0).astype(int)
    vill["row"], vill["col"] = lat_to_row(vill.latitude), lon_to_col(vill.longitude)
    vill.to_parquet(vill_path, index=False)
vill = pd.read_parquet(vill_path)
print(f"{len(vill):,} populated places; {(vill.population > 0).sum():,} with a population figure")
vill.sort_values("population", ascending=False).head()

## Done. Hand-over
Files in `ignis_earth_data/` (in your Google Drive):

| File | Used by |
|---|---|
| `celldays.parquet` | **02_model**: one row per candidate cell per fire-day, all features + `fire_tomorrow` |
| `fire_cells.parquet` | 02_model (spread-direction metric) and 03_demo (maps, the "reveal") |
| `weather_daily.parquet` | 03_demo (wind arrow, gate reasons) |
| `cell_elevation.parquet`, `villages.parquet` | 03_demo |

**Known simplifications to state in "How it works":** weather comes from the nearest 0.5° ERA5 point, not the cell itself; `days_since_rain` counts from May 15 at most; the `type` filter is skipped for years that came through the API; S-NPP only (NOAA-20 left out so every season uses the same sensor).

**Sunday add-ons (only after the model runs end to end):** CORINE land cover (`frac_conifer`, `frac_maquis`, `frac_phrygana_grass`, `frac_crop`, `frac_built`) and `rh_spotting`; Fire Weather Index (B4).